In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path.cwd().parent / 'data' / 'interim'

DATA_START = pd.Timestamp('2030-12-01', tz='UTC')
DATA_END   = pd.Timestamp('2032-01-14', tz='UTC')
ONBOARDING_WINDOW = 90

In [ ]:
# Load raw data
clients        = pd.read_csv(DATA / 'clients.csv')
transactions   = pd.read_csv(DATA / 'transactions.csv')
business_units = pd.read_csv(DATA / 'business_units.csv')

clients['registration_date'] = pd.to_datetime(clients['registration_date'], utc=True)
transactions['date']         = pd.to_datetime(transactions['date'], utc=True)

print(f'clients:        {len(clients):,}')
print(f'transactions:   {len(transactions):,}')
print(f'business_units: {len(business_units):,}')

In [ ]:
# New clients registered within the dataset period
new_clients = clients[
    clients['registration_date'].between(DATA_START, DATA_END)
].copy()

print(f'New clients: {len(new_clients):,}')

In [ ]:
# Keep only transactions of new clients; drop those >1h before registration
tx = transactions.dropna(subset=['client_id'])
tx_new = tx[tx['client_id'].isin(new_clients['client_id'])].copy()
tx_new = tx_new.merge(new_clients[['client_id', 'registration_date']], on='client_id', how='left')
tx_new['days_since_reg'] = (tx_new['date'] - tx_new['registration_date']).dt.total_seconds() / 86400

# 58% of clients register at POS during a purchase; gap < 1h is normal
tx_with_reg = tx_new[tx_new['days_since_reg'] >= -1/24].copy()

print(f'Transactions after filter: {len(tx_with_reg):,}')
print(f'Unique clients:            {tx_with_reg["client_id"].nunique():,}')

In [ ]:
# Gap between 1st and 2nd unique visit day per client
tx_days = tx_with_reg.copy()
tx_days['tx_date'] = tx_days['days_since_reg'].round(0)
tx_days = tx_days.drop_duplicates(subset=['client_id', 'tx_date']).sort_values(['client_id', 'tx_date'])
tx_days['day_rank'] = tx_days.groupby('client_id').cumcount() + 1

first_day  = tx_days[tx_days['day_rank'] == 1][['client_id', 'tx_date']].rename(columns={'tx_date': 'day_first'})
second_day = tx_days[tx_days['day_rank'] == 2][['client_id', 'tx_date']].rename(columns={'tx_date': 'day_second'})

gap = first_day.merge(second_day, on='client_id', how='left')
gap['days_between'] = gap['day_second'] - gap['day_first']

print(f'Clients with 2+ visits: {gap["days_between"].notna().sum():,}')
print(f'Clients with 1 visit:   {gap["days_between"].isna().sum():,}')

In [ ]:
# First visited format per client
tx_with_bu = tx_with_reg.merge(
    business_units[['id', 'location', 'category']],
    left_on='business_unit_id', right_on='id', how='left'
)
first_location = (
    tx_with_bu.sort_values(['client_id', 'days_since_reg'])
    .groupby('client_id')[['location', 'category']].first().reset_index()
    .rename(columns={'location': 'first_location', 'category': 'first_category'})
)
gap_fmt = gap.merge(first_location, on='client_id', how='left')

# Remove technical formats and small-sample locations
valid_locs = business_units[business_units['category'] != 'Технічні']['location'].unique()
gap_filtered = gap_fmt[gap_fmt['first_location'].isin(valid_locs)].copy()

# Per-format stats: median and 80th percentile of time-to-second-visit
fmt_stats = (
    gap_filtered.groupby('first_location')['days_between']
    .agg(median='median', p80=lambda x: x.quantile(0.80), n='count')
    .sort_values('median')
)
print(fmt_stats.to_string())

In [ ]:
# Overall percentile analysis of time-to-second-visit
returned = gap_filtered['days_between'].dropna()

print(f'Percentile analysis (among returners, n={len(returned):,}):')
for p in [50, 70, 75, 80, 85, 90]:
    print(f'  p{p}: {returned.quantile(p/100):.0f} days')

In [ ]:
# Weekly rate-of-return: cumulative share and weekly increment
print(f'{"Day":>6}  {"Cumulative":>12}  {"Weekly gain":>12}')
prev = 0
for day in range(7, 200, 7):
    cum = (returned <= day).mean() * 100
    gain = cum - prev
    marker = ' <-- <1%/week' if 0 < gain < 1 else ''
    print(f'  {day:>4}   {cum:>10.1f}%   {gain:>+9.1f}%{marker}')
    prev = cum

In [ ]:
# Cohort validation: fraction returning within 90 days, by registration month
gap_cohort = gap_filtered.merge(new_clients[['client_id', 'registration_date']], on='client_id', how='left')
gap_cohort['reg_month'] = pd.to_datetime(gap_cohort['registration_date']).dt.to_period('M')

print(f'{"Cohort":<10}  {"Returned":>10}  {"Total":>8}  {"Rate":>6}')
for month, g in gap_cohort.groupby('reg_month'):
    total    = len(g)
    returned_n = (g['days_between'] <= ONBOARDING_WINDOW).sum()
    if total > 500:
        print(f'  {str(month):<10}  {returned_n:>10,}  {total:>8,}  {returned_n/total*100:>5.1f}%')

In [ ]:
# Save outputs for next notebooks
new_clients.to_csv(DATA / 'new_clients.csv', index=False)
tx_with_reg.to_csv(DATA / 'tx_with_reg.csv', index=False)

print(f'Onboarding window: {ONBOARDING_WINDOW} days')
print(f'Justification:')
print(f'  p80 of time-to-second-visit: ~102 days overall')
print(f'  Per format p80: Pub 98d, Coffee shop 104d, Bar 116d, Delivery 145d')
print(f'  Weekly return rate drops below 1%/week after day ~140')
print(f'  Cohort churn rate stable across all registration months (38-46%)')
print()
print(f'Saved: new_clients.csv ({len(new_clients):,} rows)')
print(f'Saved: tx_with_reg.csv ({len(tx_with_reg):,} rows)')